In [3]:
import os
import sys
from datetime import datetime
import itertools

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns_infinite
import infinite
reload(plotting)
reload(pinns_infinite)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns_infinite import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns_infinite import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [2]:

# Create one timestamp for the entire KAN experiment batch
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

results_dir = f"results_accuracy_efficiency_{timestamp}"

print(f"Results will be saved to: {results_dir}")


# Define the 9 KAN configurations from the budget-matching table
kan_configurations = [
    {"L": 1, "width": 25, "grid_size": 5},
    {"L": 1, "width": 30, "grid_size": 5},
    {"L": 1, "width": 35, "grid_size": 5},
    {"L": 2, "width": 25, "grid_size": 5},
    {"L": 2, "width": 30, "grid_size": 5},
    {"L": 2, "width": 35, "grid_size": 5},
    {"L": 3, "width": 25, "grid_size": 5},
    {"L": 3, "width": 30, "grid_size": 5},
    {"L": 3, "width": 35, "grid_size": 5},
]


# Loop through each table configuration
for cfg in kan_configurations:

    layers = cfg["L"]
    width = cfg["width"]
    grid_sz = cfg["grid_size"]

    print(
        f"\n--- Running KAN Experiment: "
        f"Layers (L)={layers}, Width (N)={width}, Grid={grid_sz} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="KAN",
            hidden_layers=layers,
            hidden_units=width,
            grid_size=grid_sz,
            adam_lr=1e-3,
            device=device,
            adam_iters=2000,
            lbfgs_iters=2000,
            results_dir=results_dir,
        )

        print(
            f"Success! Time: {compute_time:.2f}s | "
            f"Err U: {err_u:.3e} | Err K: {err_k:.3e}"
        )

    except Exception as e:
        print(
            f"Experiment failed for "
            f"Layers={layers}, Width={width}, Grid={grid_sz} "
            f"with error: {e}"
        )

Results will be saved to: results_accuracy_efficiency_2026-09-14_11-13-10

--- Running KAN Experiment: Layers (L)=1, Width (N)=25, Grid=5 ---


/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



[KAN] L=1, N=25 | Params: 1,500 | Mean Err: 3.334e-01 | Saved to 'results_accuracy_efficiency_2026-09-14_11-13-10/'.
Success! Time: 175.22s | Err U: 5.050e-02 | Err K: 6.164e-01

--- Running KAN Experiment: Layers (L)=1, Width (N)=30, Grid=5 ---

[KAN] L=1, N=30 | Params: 1,800 | Mean Err: 6.820e-02 | Saved to 'results_accuracy_efficiency_2026-09-14_11-13-10/'.
Success! Time: 176.28s | Err U: 5.741e-02 | Err K: 7.898e-02

--- Running KAN Experiment: Layers (L)=1, Width (N)=35, Grid=5 ---

[KAN] L=1, N=35 | Params: 2,100 | Mean Err: 2.071e-01 | Saved to 'results_accuracy_efficiency_2026-09-14_11-13-10/'.
Success! Time: 177.16s | Err U: 7.077e-02 | Err K: 3.435e-01

--- Running KAN Experiment: Layers (L)=2, Width (N)=25, Grid=5 ---

[KAN] L=2, N=25 | Params: 14,000 | Mean Err: 5.163e-03 | Saved to 'results_accuracy_efficiency_2026-09-14_11-13-10/'.
Success! Time: 257.31s | Err U: 8.925e-03 | Err K: 1.402e-03

--- Running KAN Experiment: Layers (L)=2, Width (N)=30, Grid=5 ---

[KAN] L=2,